# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined using a [Croissant](https://mlcommons.org/croissant/) schema and is accessible via a public URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata overview
metadata = dataset.metadata
print("Dataset name:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))
print("Citation:", getattr(metadata, 'citeAs', None))
print("Published:", getattr(metadata, 'datePublished', None))

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their Croissant `@id` field.

In [ ]:
# List all record sets and their fields, referencing by @id
print("Available record sets (@id and name):")
recordsets = dataset.record_sets
for rs in recordsets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    print("  Fields (by @id):")
    for f in rs.fields:
        print(f"    - {f.id}: {f.name} (dataType: {getattr(f, 'data_type', None)})")
    print()
    # Show up to 2 records as sample
    print("  Example records (first 2 rows):")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(f"    {rec}")
        if i >= 1:
            break
    print("-")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Only `@id` values are used to refer to entities.

In [ ]:
# Gather all record_set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
# Load data for each record set (limit to 5000 rows for testing)
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)

# For demonstration, print column names and head for first record set
if record_set_ids:
    first_id = record_set_ids[0]
    print("Fields (@id) for record set:", first_id)
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing, and grouping data, referencing by field `@id`.

Choose a numeric field (by `@id`) and a group field from the record set.

In [ ]:
import numpy as np

# For the FAIR^2 dataset, example field @id might be 'Accession', 'MW [kDa]', etc.
# Let's try to auto-detect a numeric field and group field
if record_set_ids:
    df = dataframes[first_id]
    # Try to find likely numeric field
    numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.float32, np.int64, np.int32] or (df[c].dtype == object and pd.to_numeric(df[c], errors='coerce').notna().sum() > 0)]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    else:
        numeric_field = None

    group_field = None
    # Try to find a categorical field that is not numeric
    for c in df.columns:
        if df[c].dtype == object and (df[c].nunique() < df.shape[0] / 2):
            group_field = c
            break

    if numeric_field:
        print(f"Numeric field detected: {numeric_field}")
        threshold = np.nanmean(df[numeric_field]) if not np.isnan(df[numeric_field]).all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered {numeric_field} > {threshold:.2f} (n={len(filtered_df)})")
        display(filtered_df[[numeric_field]].head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Grouping if group_field exists
        if group_field:
            print(f"Grouping by {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please adjust the code to specify a field by @id.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field:
    plt.figure(figsize=(12, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {first_id}")
    plt.xlabel(numeric_field)
    plt.show()
    if norm_col in df.columns:
        plt.figure(figsize=(12, 5))
        sns.histplot(filtered_df[norm_col].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of Normalized {numeric_field} in record set {first_id}")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load, review, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` for clarity and reproducibility.

You may now tailor the data processing and analysis to your specific goals using the dataset structure discovered here.